In [38]:
!pip install folium pandas -q
import math
import pandas as pd
import folium
from IPython.display import HTML, display

# Hàm hiển thị map trong Colab
def show_map(m):
    display(HTML(m._repr_html_()))
    return m

# Hàm tính khoảng cách km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Trung tâm phân phối
distribution_center = (10.762622, 106.660172)

# Dữ liệu khách hàng mẫu
customers = pd.DataFrame([
    {"id": 1, "lat": 10.7612, "lon": 106.6571},
    {"id": 2, "lat": 10.7629, "lon": 106.6630},
    {"id": 3, "lat": 10.7655, "lon": 106.6601},
    {"id": 4, "lat": 10.7595, "lon": 106.6549},
    {"id": 5, "lat": 10.7660, "lon": 106.6660},
    {"id": 6, "lat": 10.7582, "lon": 106.6615},
    {"id": 7, "lat": 10.7638, "lon": 106.6565},
    {"id": 8, "lat": 10.7607, "lon": 106.6645},
])

# Tính khoảng cách và phân vùng
customers["distance_km"] = customers.apply(
    lambda r: haversine_km(distribution_center[0], distribution_center[1], r["lat"], r["lon"]),
    axis=1
)

def assign_zone(d):
    if d <= 3:
        return "<= 3 km"
    elif d <= 5:
        return "<= 5 km"
    elif d <= 10:
        return "<= 10 km"
    else:
        return "> 10 km"

customers["zone"] = customers["distance_km"].apply(assign_zone)

print("Bảng phân vùng khách hàng:")
print(customers[["id", "distance_km", "zone"]])

# Tạo bản đồ
m = folium.Map(location=distribution_center, zoom_start=13)

# Marker trung tâm
folium.Marker(
    location=distribution_center,
    popup="Trung tâm phân phối",
    icon=folium.Icon(color="red")
).add_to(m)

# Vẽ các vòng tròn vùng phục vụ
folium.Circle(
    location=distribution_center,
    radius=3000,
    color="green",
    fill=True,
    fill_opacity=0.1,
    popup="Vùng phục vụ 3 km"
).add_to(m)

folium.Circle(
    location=distribution_center,
    radius=5000,
    color="orange",
    fill=True,
    fill_opacity=0.1,
    popup="Vùng phục vụ 5 km"
).add_to(m)

folium.Circle(
    location=distribution_center,
    radius=10000,
    color="blue",
    fill=True,
    fill_opacity=0.1,
    popup="Vùng phục vụ 10 km"
).add_to(m)

# Thêm khách hàng lên map
for _, row in customers.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"Khách {row['id']}<br>Khoảng cách: {row['distance_km']:.2f} km<br>Vùng: {row['zone']}"
    ).add_to(m)

show_map(m)

Bảng phân vùng khách hàng:
   id  distance_km     zone
0   1     0.370968  <= 3 km
1   2     0.310470  <= 3 km
2   3     0.320116  <= 3 km
3   4     0.672448  <= 3 km
4   5     0.739188  <= 3 km
5   6     0.512658  <= 3 km
6   7     0.421970  <= 3 km
7   8     0.518848  <= 3 km


In [ ]:
from google.colab import drive
drive.mount('/content/drive')